## Import

In [4]:
import os
import mne
from mne import events_from_annotations, create_info, EpochsArray, concatenate_epochs, Epochs
import json
import pandas as pd
from pathlib import Path
import re
import matplotlib.pyplot as plt
import numpy as np
import warnings
from collections import defaultdict
from eegkit.models import (
    TaskDTO, FilterParamsDTO, TimeDomainParamsDTO, PSDParamsDTO,
    EpochParamsDTO, EpochFullParamsDTO, TableInfoDTO, EpochPSDParamsDTO
)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

from notebook_utils import reload_classes, reload_data_classes, reload_model_classes

warnings.filterwarnings("ignore", message=".*boundary.*data discontinuities.*")
warnings.filterwarnings("ignore", message="FigureCanvasAgg is non-interactive, and thus cannot be shown")

## Load

In [5]:
confirmation = input("Type '1' to reload: ")

if confirmation.lower() == '1':
    EEGSubjectData = reload_data_classes()
    release = 1
    data_dir = f'/mount/intern/NAS-public-dataset/HBN-EEG/cmi_bids_R{release}'
    subject_data = EEGSubjectData(data_dir)

In [6]:
EEGController, EEGUI = reload_classes()
controller = EEGController(subject_data)
subject_model = controller.subject_model
visualizer = controller.visualizer
data_service = controller.data_service

In [7]:
ai_service = reload_model_classes()

In [8]:
subjects = sorted(controller.list_subjects())
subject = subjects[0]
subject

'sub-NDARAC904DMU'

In [9]:
task_keys = sorted(controller.list_tasks(subject))
for (i, item) in enumerate(task_keys):
    print(i, item)

0 ('DespicableMe', None)
1 ('DiaryOfAWimpyKid', None)
2 ('FunwithFractals', None)
3 ('RestingState', None)
4 ('ThePresent', None)
5 ('contrastChangeDetection', '1')
6 ('contrastChangeDetection', '2')
7 ('contrastChangeDetection', '3')
8 ('contrastChangeDetection', 'All 3')
9 ('seqLearning8target', None)
10 ('surroundSupp', '1')
11 ('surroundSupp', '2')
12 ('surroundSupp', 'All 2')
13 ('symbolSearch', None)


## NN Class

In [10]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [11]:
num_gpus = torch.cuda.device_count()
print(f"Number of GPUs: {num_gpus}")

# List all devices with their names
for i in range(num_gpus):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# Current default device index
if torch.cuda.is_available():
    print(f"Current default device: cuda:{torch.cuda.current_device()}")
else:
    print("No CUDA-compatible GPU found.")

Number of GPUs: 0
No CUDA-compatible GPU found.


In [12]:
import torch, platform, sys
print("PyTorch:", torch.__version__)
print("CUDA build in torch:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("Python:", sys.version.split()[0], "OS:", platform.platform())

PyTorch: 2.4.1
CUDA build in torch: None
CUDA available: False
GPU count: 0
Python: 3.12.11 OS: Linux-5.15.0-46-generic-x86_64-with-glibc2.31


# SurroundSupp

In [13]:
task, run = task_keys[12]
l_freq = 3.0
h_freq = 35.0
tmin = -0.2
tmax = 2.4

task_dto = TaskDTO(subject=subject, task=task, run=run)
filter_params = FilterParamsDTO(l_freq=l_freq, h_freq=h_freq)
epoch_params = EpochParamsDTO(l_freq=l_freq, h_freq=h_freq, tmin=tmin, tmax=tmax)

task_model = subject_model.get_task(task_dto)

data_service.show_annotations(task_dto,filter_params)

{'PowerLineFrequency': 60,
 'TaskName': 'surroundSupp',
 'EEGChannelCount': 129,
 'EEGReference': 'Cz',
 'RecordingType': 'continuous',
 'RecordingDuration': 486.256,
 'SamplingFrequency': 500,
 'SoftwareFilters': 'n/a'}

In [14]:
raw = task_model.get_filtered_raw(filter_params)

Filtering raw data in 2 contiguous segments
Setting up band-pass filter from 3 - 35 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 3.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 2.00 Hz)
- Upper passband edge: 35.00 Hz
- Upper transition bandwidth: 8.75 Hz (-6 dB cutoff frequency: 39.38 Hz)
- Filter length: 825 samples (1.650 s)



In [15]:
epochs, labels = task_model.get_epochs(epoch_params)
X = epochs.get_data() 
n_epochs, n_channels, n_times = X.shape
print("Epochs shape:", X.shape)
print("Labels example:", labels[:10])

Not setting metadata
128 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 128 events and 1301 original time points ...


/mount/intern/NAS-workspace-portal/eeg2025-Vistec/eegkit/models/task_processor.py:66: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = Epochs(


0 bad epochs dropped
Epochs shape: (128, 128, 1301)
Labels example: ['bg1_fg0.0_stim2' 'bg1_fg0.3_stim3' 'bg1_fg0.6_stim1' 'bg1_fg1.0_stim2'
 'bg1_fg0.0_stim3' 'bg1_fg0.3_stim2' 'bg1_fg0.6_stim1' 'bg1_fg1.0_stim3'
 'bg1_fg0.0_stim2' 'bg1_fg0.3_stim3']


# Dependent 

## foreground classcification

In [16]:
def extract_fg(label):
    m = re.search(r'fg([0-9.]+)', label)
    return float(m.group(1)) if m else None

fg_values = np.array([extract_fg(lbl) for lbl in labels])

valid_fg = {0.0, 0.3, 0.6, 1.0}
mask = np.isin(fg_values, list(valid_fg))

X_fg = X[mask]
y_fg = fg_values[mask]
labels_fg = np.array(labels)[mask]

print("Filtered epochs shape:", X_fg.shape)
print("Filtered fg labels:", y_fg[:10])

Filtered epochs shape: (128, 128, 1301)
Filtered fg labels: [0.  0.3 0.6 1.  0.  0.3 0.6 1.  0.  0.3]


In [17]:
fg_to_class = {0.0: 0, 0.3: 1, 0.6: 2, 1.0: 3}
y_class = np.array([fg_to_class[v] for v in y_fg])

print("Classification targets (y_class):", y_class[:10])
print("Mapping:", fg_to_class)

Classification targets (y_class): [0 1 2 3 0 1 2 3 0 1]
Mapping: {0.0: 0, 0.3: 1, 0.6: 2, 1.0: 3}


In [18]:
n_epochs, n_channels, n_times = X_fg.shape

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_fg, y_class, test_size=0.2, random_state=42
)

# Convert to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)  # (batch, channels, time)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.long)

# Create DataLoaders
train_ds = TensorDataset(X_train, y_train)
test_ds  = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=32)

# Model — first conv layer in_channels should match n_channels
input_channels = n_channels
num_classes = len(np.unique(y_class))

model = ai_service.CNNLSTMDense(in_channels=input_channels, num_classes=num_classes)

print(input_channels,num_classes)

128 4


In [19]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs_num = 100

for epoch in range(epochs_num):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        out = model(xb)
        loss = loss_fn(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} loss: {total_loss/len(train_loader):.4f}")


Epoch 1 loss: 1.3879
Epoch 2 loss: 1.3806
Epoch 3 loss: 1.3985
Epoch 4 loss: 1.3888
Epoch 5 loss: 1.3808
Epoch 6 loss: 1.3699
Epoch 7 loss: 1.3712
Epoch 8 loss: 1.3670
Epoch 9 loss: 1.3565
Epoch 10 loss: 1.3319
Epoch 11 loss: 1.3262
Epoch 12 loss: 1.3443
Epoch 13 loss: 1.2964
Epoch 14 loss: 1.2670
Epoch 15 loss: 1.3074
Epoch 16 loss: 1.2782
Epoch 17 loss: 1.2879
Epoch 18 loss: 1.2797
Epoch 19 loss: 1.2573
Epoch 20 loss: 1.2380
Epoch 21 loss: 1.1704
Epoch 22 loss: 1.2882
Epoch 23 loss: 1.2114
Epoch 24 loss: 1.2027
Epoch 25 loss: 1.1649
Epoch 26 loss: 1.1930
Epoch 27 loss: 1.2482
Epoch 28 loss: 1.1408
Epoch 29 loss: 1.1670
Epoch 30 loss: 1.0644
Epoch 31 loss: 1.0857
Epoch 32 loss: 1.1328
Epoch 33 loss: 1.1054
Epoch 34 loss: 1.0258
Epoch 35 loss: 1.0150
Epoch 36 loss: 0.9834
Epoch 37 loss: 1.0802
Epoch 38 loss: 1.0657
Epoch 39 loss: 0.9428
Epoch 40 loss: 1.0415
Epoch 41 loss: 0.9285
Epoch 42 loss: 0.8476
Epoch 43 loss: 0.9511
Epoch 44 loss: 0.8510
Epoch 45 loss: 0.9172
Epoch 46 loss: 0.95

In [20]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for xb, yb in test_loader:
        out = model(xb)
        pred = torch.argmax(out, dim=1)
        correct += (pred == yb).sum().item()
        total += yb.size(0)
print("Test accuracy:", correct / total)


Test accuracy: 0.15384615384615385


# Contrast Change Detection classcification